In [5]:
import glob
import pytesseract

In [7]:
img = glob.glob('./mutasi-example.jpg')

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

for i,image in enumerate(img):
    text = pytesseract.image_to_string(image,lang='eng')
    print(text)

oes

1100-00-020

Tanggal Tp
Salo Awal:
arpo.s

sen014

aaszo14 si)

a7/2014 PI
s2y2014 si

o
aag2014 ss)
ss07014 R
‘aldo Awal:
Saldo Akhir
1200-00-010
Tanggal Tp
Salo Awal:
anozo14 R
algyzo14 R
anszo4
28/2014 PI
4292014 D
1302014 PI
3)
R
1ayz14 R
o
o
o
Saldo Awal:
Saldo Akhir
1200-00-021
Tanggal Tp
Salo Awal:
anszo4
‘Saldo Awal:
Saldo Akhir

25 March, 2014

Dagang Distribusi
Buku Besar - Mutasi

Wednesday, January 01, 2014 - Friday, January 31, 2014

Kas
No.Ref. _Keterangan Debet
cD000001_Pengeluaran, Umum untuk Bulan Desember 2013,
cD000002_Pengeluaran,AktivaMakmur Pembelian AirCon
cD000004 —_Pengeluaran, Umum Pembelian Meja Rapat dan Printer
Wama
cD000005_Pengeluaran, Umum
‘0000000 ——Penjualan, Persada 2,900 00000
‘00000003 Pembelian, Atv Makmur
(00000005 —_Penjualan, Maharaja 7,281,75000
cb000003.—_Pengeluaran, Umum
‘00000006 _Penjualan, Pertini Agung 8,500,000.00
crO00004 —_—Penerimaan dari Persada 390,000
500,000,000.00 38,171,750.00
212,601,750.00 287,398, 250.00
Bank
No.Ref. _K

In [10]:
import keras_ocr
import glob
import pandas as pd
import matplotlib.pyplot as plt

ImportError: cannot import name '_center' from 'numpy._core.umath' (c:\Users\Michael\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\_core\umath.py)

In [ ]:
img = glob.glob('./mutasi-example.jpg')

pipeline = keras_ocr.pipeline.Pipeline()
results = pipeline.recognize(
    [img[0]],
    
)

df = pd.DataFrame(results[0],columns=["text","bbox"])
pd.set_option('display,max_column')

NameError: name 'keras_ocr' is not defined

In [ ]:
import cv2
import re
import json
import numpy as np
from PIL import Image
from paddleocr import PaddleOCR
import pytesseract
from transformers import pipeline

# ============================================================
# OCR ENGINE
# ============================================================

class InvoiceOCR:
    def __init__(self):
        self.ocr = PaddleOCR(
            use_angle_cls=True,
            lang='en'
        )

    def preprocess(self, image_path):
        image = cv2.imread(image_path)

        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Denoise
        gray = cv2.fastNlMeansDenoising(gray)

        # Threshold
        thresh = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )[1]

        return thresh

    def extract_text_paddle(self, image_path):
        result = self.ocr.ocr(image_path, cls=True)

        full_text = []

        for line in result:
            for word in line:
                text = word[1][0]
                full_text.append(text)

        return "\n".join(full_text)

    def extract_text_tesseract(self, image_path):
        processed = self.preprocess(image_path)

        text = pytesseract.image_to_string(processed)

        return text


# ============================================================
# INVOICE PARSER
# ============================================================

class InvoiceParser:

    def __init__(self):
        pass

    def parse_invoice(self, text):

        data = {
            "invoice_number": None,
            "invoice_date": None,
            "vendor": None,
            "subtotal": None,
            "tax": None,
            "total": None,
            "currency": None,
            "items": []
        }

        # ----------------------------------------------------
        # Invoice Number
        # ----------------------------------------------------
        invoice_patterns = [
            r"Invoice\s*#?\s*[:\-]?\s*([A-Z0-9\-]+)",
            r"Invoice Number\s*[:\-]?\s*([A-Z0-9\-]+)"
        ]

        for pattern in invoice_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                data["invoice_number"] = match.group(1)
                break

        # ----------------------------------------------------
        # Date
        # ----------------------------------------------------
        date_patterns = [
            r"(\d{2}/\d{2}/\d{4})",
            r"(\d{4}-\d{2}-\d{2})",
            r"(\d{2}-\d{2}-\d{4})"
        ]

        for pattern in date_patterns:
            match = re.search(pattern, text)
            if match:
                data["invoice_date"] = match.group(1)
                break

        # ----------------------------------------------------
        # Vendor (Usually first line)
        # ----------------------------------------------------
        lines = text.split("\n")

        cleaned = [l.strip() for l in lines if l.strip()]

        if len(cleaned) > 0:
            data["vendor"] = cleaned[0]

        # ----------------------------------------------------
        # Total
        # ----------------------------------------------------
        total_patterns = [
            r"Total\s*[:\-]?\s*\$?\s*([\d,]+\.\d{2})",
            r"Grand Total\s*[:\-]?\s*\$?\s*([\d,]+\.\d{2})"
        ]

        for pattern in total_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                data["total"] = match.group(1)
                break

        # ----------------------------------------------------
        # Tax
        # ----------------------------------------------------
        tax_match = re.search(
            r"Tax\s*[:\-]?\s*\$?\s*([\d,]+\.\d{2})",
            text,
            re.IGNORECASE
        )

        if tax_match:
            data["tax"] = tax_match.group(1)

        # ----------------------------------------------------
        # Currency Detection
        # ----------------------------------------------------
        if "$" in text:
            data["currency"] = "USD"
        elif "Rp" in text:
            data["currency"] = "IDR"
        elif "€" in text:
            data["currency"] = "EUR"

        # ----------------------------------------------------
        # Line Item Detection
        # Example:
        # Apple x2 10.00
        # ----------------------------------------------------
        item_pattern = r"([A-Za-z0-9\s]+)\s+x?(\d+)\s+([\d,]+\.\d{2})"

        matches = re.findall(item_pattern, text)

        for item in matches:
            name, qty, price = item

            data["items"].append({
                "name": name.strip(),
                "quantity": int(qty),
                "price": float(price.replace(",", ""))
            })

        return data


# ============================================================
# LAYOUT-AWARE TRANSFORMER MODEL (OPTIONAL AI PARSER)
# ============================================================

class AIInvoiceExtractor:
    """
    Uses HuggingFace transformers for document understanding.
    Better for real-world invoices.
    """

    def __init__(self):

        self.nlp = pipeline(
            "document-question-answering",
            model="impira/layoutlm-document-qa"
        )

    def ask(self, image_path, question):

        result = self.nlp(
            image=image_path,
            question=question
        )

        return result


# ============================================================
# MAIN
# ============================================================

if __name__ == "__main__":

    IMAGE_PATH = "invoice.jpg"

    ocr_engine = InvoiceOCR()

    # --------------------------------------------------------
    # OCR Extraction
    # --------------------------------------------------------

    print("Running OCR...\n")

    text = ocr_engine.extract_text_paddle(IMAGE_PATH)

    print("========== RAW TEXT ==========\n")
    print(text)

    # --------------------------------------------------------
    # Parse Invoice
    # --------------------------------------------------------

    parser = InvoiceParser()

    invoice_json = parser.parse_invoice(text)

    print("\n========== JSON OUTPUT ==========\n")

    print(json.dumps(invoice_json, indent=4))

    # --------------------------------------------------------
    # OPTIONAL AI QA Extraction
    # --------------------------------------------------------

    try:

        ai_extractor = AIInvoiceExtractor()

        print("\n========== AI EXTRACTION ==========\n")

        questions = [
            "What is the invoice number?",
            "What is the total amount?",
            "What is the invoice date?",
            "Who is the vendor?"
        ]

        for q in questions:

            answer = ai_extractor.ask(IMAGE_PATH, q)

            print(f"{q}")
            print(answer)
            print()

    except Exception as e:
        print("AI extractor skipped:", e)